# Regression Benchmark
Unified pipeline to train, evaluate, and compare multiple sklearn regressors side-by-side.

**Metrics collected**
| Metric | Description |
|---|---|
| MAE | Mean Absolute Error |
| MSE | Mean Squared Error |
| RMSE | Root Mean Squared Error |
| R² | Coefficient of Determination |
| Train time | Wall-clock fit duration (seconds) |
| Predict time | Wall-clock inference duration (seconds) |
| Peak memory | `tracemalloc` peak during training (KiB) |

In [1]:
import warnings, sys, os
sys.path.insert(0, "/home/claude/benchmarking")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from metrics_utils import (BenchmarkResult, highlight_best, memory_tracker,
                            prepare_split, print_summary_table,
                            results_to_dataframe, timer)

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "#f8f9fb",
    "axes.grid": True, "grid.color": "white", "grid.linewidth": 1.2,
    "axes.spines.top": False, "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")
print("✓ Imports OK")

✓ Imports OK


## RegressionBenchmark class

In [2]:
class RegressionBenchmark:
    def __init__(self, X, y, dataset_name="Dataset", test_size=0.2, random_state=42):
        self.dataset_name = dataset_name
        self.X_train, self.X_test, self.y_train, self.y_test = prepare_split(
            np.asarray(X, dtype=float), np.asarray(y, dtype=float),
            test_size=test_size, random_state=random_state)
        self._models = []
        self.results = []

    def add_model(self, name, model):
        self._models.append((name, model))
        return self

    def run(self):
        self.results.clear()
        print(f"\n🔬  Benchmarking on '{self.dataset_name}'  "
              f"({self.X_train.shape[0]} train / {self.X_test.shape[0]} test samples)\n")
        for name, model in self._models:
            result = self._evaluate(name, model)
            self.results.append(result)
            print(f"  ✓  {name:<35}  R²={result.metrics['r2']:.4f}  "
                  f"RMSE={result.metrics['rmse']:.4f}  train={result.train_time_s:.3f}s")
        return self

    def print_summary(self):
        df = results_to_dataframe(self.results)
        highlighted = highlight_best(df, ["r2"],
            ["mae","mse","rmse","train_time_s","predict_time_s","peak_memory_kb"])
        print_summary_table(highlighted, title=f"Regression — {self.dataset_name}")
        return df

    def save_results(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        df = results_to_dataframe(self.results)
        df.to_csv(path)
        print(f"  📄  Saved → {path}")
        return df

    def _evaluate(self, name, model):
        result = BenchmarkResult(model_name=name, task="regression")
        with memory_tracker() as mem, timer() as t:
            model.fit(self.X_train, self.y_train)
        result.train_time_s   = t["elapsed"]
        result.peak_memory_kb = mem["peak_kb"]
        with timer() as t:
            y_pred = model.predict(self.X_test)
        result.predict_time_s = t["elapsed"]
        mae  = mean_absolute_error(self.y_test, y_pred)
        mse  = mean_squared_error(self.y_test, y_pred)
        result.metrics.update({"mae": mae, "mse": mse,
                                "rmse": float(np.sqrt(mse)),
                                "r2": r2_score(self.y_test, y_pred)})
        return result

print("✓ RegressionBenchmark defined")

✓ RegressionBenchmark defined


## Models registered for both datasets

In [3]:
from sklearn.datasets import load_diabetes, make_regression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

MODELS_REG = [
    ("Linear Regression",      LinearRegression()),
    ("Ridge",                  Ridge(alpha=1.0)),
    ("Lasso",                  Lasso(alpha=0.1, max_iter=5000)),
    ("ElasticNet",             ElasticNet(max_iter=5000)),
    ("Decision Tree",          DecisionTreeRegressor(random_state=42)),
    ("Random Forest",          RandomForestRegressor(n_estimators=100, random_state=42)),
    ("Gradient Boosting",      GradientBoostingRegressor(n_estimators=100, random_state=42)),
    ("K-Nearest Neighbours",   KNeighborsRegressor(n_neighbors=5)),
    ("Support Vector Machine", SVR(kernel="rbf")),
]
print("✓ Models defined")

✓ Models defined


## Dataset 1 — Diabetes

In [4]:
X, y = load_diabetes(return_X_y=True)
bench_diab = RegressionBenchmark(X, y, dataset_name="Diabetes")
for name, model in MODELS_REG:
    bench_diab.add_model(name, model)
bench_diab.run()
df_diab = bench_diab.print_summary()
bench_diab.save_results("sample_results/regression_diabetes.csv")


🔬  Benchmarking on 'Diabetes'  (353 train / 89 test samples)

  ✓  Linear Regression                    R²=0.4526  RMSE=53.8534  train=0.005s
  ✓  Ridge                                R²=0.4192  RMSE=55.4745  train=0.004s
  ✓  Lasso                                R²=0.4719  RMSE=52.8980  train=0.003s
  ✓  ElasticNet                           R²=-0.0025  RMSE=72.8781  train=0.003s
  ✓  Decision Tree                        R²=0.0607  RMSE=70.5464  train=0.010s


  ✓  Random Forest                        R²=0.4428  RMSE=54.3324  train=1.112s


  ✓  Gradient Boosting                    R²=0.4529  RMSE=53.8371  train=0.275s
  ✓  K-Nearest Neighbours                 R²=0.4302  RMSE=54.9461  train=0.002s
  ✓  Support Vector Machine               R²=0.1821  RMSE=65.8277  train=0.008s

──────────────────────────────────────────────────────────────────────────────────────────
  Regression — Diabetes
──────────────────────────────────────────────────────────────────────────────────────────
                       train_time_s predict_time_s peak_memory_kb        mae          mse      rmse        r2
model                                                                                                        
Linear Regression            0.0051       0.000169           76.9    42.7941    2900.1936   53.8534    0.4526
Ridge                        0.0039       0.000135           61.5    46.1389    3077.4159   55.4745    0.4192
Lasso                        0.0034       0.000162           62.9    42.8544  2798.1935 ★  52.898 ★  0.4719 ★
Ela

,train_time_s,predict_time_s,peak_memory_kb,mae,mse,rmse,r2
model,,,,,,,
Linear Regression,0.0051,0.000169,76.9,42.7941,2900.1936,53.8534,0.4526
Ridge,0.0039,0.000135,61.5,46.1389,3077.4159,55.4745,0.4192
Lasso,0.0034,0.000162,62.9,42.8544,2798.1935,52.8980,0.4719
ElasticNet,0.0025,0.000133,61.7,63.7059,5311.2128,72.8781,-0.0025
Decision Tree,0.0102,0.000192,33.5,54.5281,4976.7978,70.5464,0.0607
Random Forest,1.1125,0.009811,102.4,44.0530,2952.0106,54.3324,0.4428
Gradient Boosting,0.2753,0.000649,96.4,44.6033,2898.4367,53.8371,0.4529
K-Nearest Neighbours,0.0023,0.001181,16.3,42.7708,3019.0755,54.9461,0.4302
Support Vector Machine,0.0076,0.001612,43.6,56.0237,4333.2860,65.8277,0.1821


## Dataset 2 — Synthetic (500 samples, 20 features)

In [5]:
X2, y2 = make_regression(n_samples=500, n_features=20, noise=10.0, random_state=42)
bench_syn = RegressionBenchmark(X2, y2, dataset_name="Synthetic")
for name, model in MODELS_REG:
    bench_syn.add_model(name, model)
bench_syn.run()
df_syn = bench_syn.print_summary()
bench_syn.save_results("sample_results/regression_synthetic.csv")


🔬  Benchmarking on 'Synthetic'  (400 train / 100 test samples)

  ✓  Linear Regression                    R²=0.9948  RMSE=9.8418  train=0.003s
  ✓  Ridge                                R²=0.9947  RMSE=9.9281  train=0.004s
  ✓  Lasso                                R²=0.9949  RMSE=9.7667  train=0.003s
  ✓  ElasticNet                           R²=0.8476  RMSE=53.3415  train=0.002s
  ✓  Decision Tree                        R²=0.5597  RMSE=90.6727  train=0.010s


  ✓  Random Forest                        R²=0.7522  RMSE=68.0197  train=1.265s


  ✓  Gradient Boosting                    R²=0.8806  RMSE=47.2225  train=0.388s
  ✓  K-Nearest Neighbours                 R²=0.5531  RMSE=91.3485  train=0.001s
  ✓  Support Vector Machine               R²=0.0498  RMSE=133.2041  train=0.009s

──────────────────────────────────────────────────────────────────────────────────────────
  Regression — Synthetic
──────────────────────────────────────────────────────────────────────────────────────────
                       train_time_s predict_time_s peak_memory_kb       mae         mse      rmse        r2
model                                                                                                      
Linear Regression            0.0034        0.00026          153.5    8.0437     96.8614    9.8418    0.9948
Ridge                        0.0039     0.000125 ★          131.8    8.1279      98.568    9.9281    0.9947
Lasso                        0.0026       0.000158          131.8  8.0182 ★   95.3875 ★  9.7667 ★  0.9949 ★
ElasticNet 

,train_time_s,predict_time_s,peak_memory_kb,mae,mse,rmse,r2
model,,,,,,,
Linear Regression,0.0034,0.000260,153.5,8.0437,96.8614,9.8418,0.9948
Ridge,0.0039,0.000125,131.8,8.1279,98.5680,9.9281,0.9947
Lasso,0.0026,0.000158,131.8,8.0182,95.3875,9.7667,0.9949
ElasticNet,0.0024,0.000147,131.9,41.4719,2845.3152,53.3415,0.8476
Decision Tree,0.0100,0.000171,50.3,69.2163,8221.5327,90.6727,0.5597
Random Forest,1.2648,0.009179,118.1,55.1397,4626.6794,68.0197,0.7522
Gradient Boosting,0.3883,0.000698,109.6,36.9942,2229.9657,47.2225,0.8806
K-Nearest Neighbours,0.0010,0.022833,3.6,72.6967,8344.5536,91.3485,0.5531
Support Vector Machine,0.0091,0.002130,78.2,103.0090,17743.3417,133.2041,0.0498


## Visualizations

In [6]:
def grouped_bar_reg(df, metrics, title, path):
    models = df.index.tolist()
    x = np.arange(len(models))
    width = 0.8 / len(metrics)
    fig, ax = plt.subplots(figsize=(max(10, len(models)*1.5), 5), dpi=130)
    for i, metric in enumerate(metrics):
        vals = df[metric].values
        bars = ax.bar(x + (i - len(metrics)/2 + 0.5)*width, vals,
                      width*0.85, label=metric.upper() if metric!="r2" else "R²",
                      color=PALETTE[i % len(PALETTE)], zorder=3)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height() + 0.003*abs(ax.get_ylim()[1]),
                    f"{val:.2f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=30, ha="right")
    ax.set_title(title); ax.legend(loc="upper right"); fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches="tight"); plt.show()
    print(f"  🖼  {path}")

def heatmap_reg(df, metrics, title, path):
    sub  = df[metrics].copy()
    norm = (sub - sub.min()) / (sub.max() - sub.min() + 1e-9)
    for col in ["mae","mse","rmse","train_time_s","predict_time_s","peak_memory_kb"]:
        if col in norm.columns: norm[col] = 1 - norm[col]
    fig, ax = plt.subplots(figsize=(len(metrics)*1.3+2, len(sub)*0.6+2), dpi=130)
    sns.heatmap(norm, annot=sub.round(3), fmt=".3f", cmap="YlOrRd_r",
                linewidths=0.4, linecolor="white", ax=ax,
                cbar_kws={"label":"Normalised rank (green=better)"}, annot_kws={"size":9})
    ax.set_xticklabels([m.upper() if m!="r2" else "R²" for m in metrics], rotation=30, ha="right")
    ax.set_title(title, pad=12); fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches="tight"); plt.show()
    print(f"  🖼  {path}")

print("✓ Plot helpers ready")

✓ Plot helpers ready


In [7]:
# Diabetes — bar chart
grouped_bar_reg(df_diab, ["mae","rmse","r2"],
                "Regression metrics — Diabetes", "sample_results/reg_diab_metrics.png")

  🖼  sample_results/reg_diab_metrics.png


In [8]:
# Diabetes — heatmap
heatmap_reg(df_diab, ["mae","mse","rmse","r2"],
            "Metric heatmap — Diabetes", "sample_results/reg_diab_heatmap.png")

  🖼  sample_results/reg_diab_heatmap.png


In [9]:
# Synthetic — bar chart
grouped_bar_reg(df_syn, ["mae","rmse","r2"],
                "Regression metrics — Synthetic", "sample_results/reg_syn_metrics.png")

  🖼  sample_results/reg_syn_metrics.png


In [10]:
# Synthetic — heatmap
heatmap_reg(df_syn, ["mae","mse","rmse","r2"],
            "Metric heatmap — Synthetic", "sample_results/reg_syn_heatmap.png")

  🖼  sample_results/reg_syn_heatmap.png


In [11]:
# Speed vs R² scatter (Diabetes)
fig, ax = plt.subplots(figsize=(8,5), dpi=130)
rmse_inv = 1 / (df_diab["rmse"].values + 1e-6)
sizes = rmse_inv / rmse_inv.max() * 400 + 50
ax.scatter(df_diab["train_time_s"], df_diab["r2"], s=sizes,
           c=range(len(df_diab)), cmap="tab10", alpha=0.85, edgecolors="white", linewidths=0.8, zorder=3)
for model, row in df_diab.iterrows():
    ax.annotate(model, (row["train_time_s"], row["r2"]),
                textcoords="offset points", xytext=(8,4), fontsize=8)
ax.set_xlabel("Training time (s)"); ax.set_ylabel("R²")
ax.set_title("Speed vs R² — Diabetes")
ax.text(0.02, 0.02, "Marker size ∝ 1/RMSE  (larger = lower error)",
        transform=ax.transAxes, fontsize=8, color="#666")
fig.tight_layout()
fig.savefig("sample_results/reg_diab_speed.png", dpi=130, bbox_inches="tight")
plt.show(); print("  🖼  sample_results/reg_diab_speed.png")

  🖼  sample_results/reg_diab_speed.png


In [12]:
# Performance overhead bars (Diabetes)
overhead_cols = ["train_time_s","predict_time_s","peak_memory_kb"]
labels = {"train_time_s":"Train time (s)","predict_time_s":"Predict time (s)","peak_memory_kb":"Peak memory (KB)"}
fig, axes = plt.subplots(1, 3, figsize=(15, max(4, len(df_diab)*0.55+1.5)), dpi=130)
y = np.arange(len(df_diab))
for ax, col in zip(axes, overhead_cols):
    vals = df_diab[col].values
    bars = ax.barh(y, vals, color=PALETTE[overhead_cols.index(col)], alpha=0.85, zorder=3)
    ax.set_yticks(y); ax.set_yticklabels(df_diab.index); ax.set_xlabel(labels[col])
    ax.set_title(labels[col]); ax.invert_yaxis()
    for bar, val in zip(bars, vals):
        ax.text(val*1.01, bar.get_y()+bar.get_height()/2, f"{val:.4f}", va="center", fontsize=8)
fig.suptitle("Performance overhead — Diabetes", fontsize=13); fig.tight_layout()
fig.savefig("sample_results/reg_diab_overhead.png", dpi=130, bbox_inches="tight")
plt.show(); print("  🖼  sample_results/reg_diab_overhead.png")

  🖼  sample_results/reg_diab_overhead.png
